### THE CASE OF FLIXBUS: POSITIONING STRATEGY AND PRICE ALLOCATION IN LONG DISTANCE BUS SERVICE
###### Merlin Mirley Moreno Palacios 
###### Date: 04/2026


In [7]:
# Data manipulation
import pandas as pd
import numpy as np
# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

#### 1. Data loading and exploratory inspection

I start by loading the dataset from an Excel source and performing an initial inspection. This step helps verify the structure of the data, detect missing values, and understand the distributions of the main variables before applying any transformations.

To do this, I examine the dataset structure, variable types, and summary statistics.

In [ ]:
# Load dataset
df_final = pd.read_excel("DF.xlsx")

# Initial inspection
print(df_final.info())
display(df_final.describe(include="all"))
display(df_final.head())

## 2. Fare variables and data cleaning

In the dataset, two different fare measures are constructed in order to capture distinct pricing scenarios observed on the FlixBus platform.

The variable *fare1* represents the price associated with the first available seat for a given query. This variable approximates the lowest fare offered at a given point in time and is therefore used as the main dependent variable in the empirical analysis. It reflects the initial price level faced by consumers when searching for tickets.

In contrast, *fare2* represents an alternative pricing measure that accounts for situations in which the booking request does not necessarily correspond to the first available seat. This occurs particularly in censored observations, where the platform returns valid prices for all seat requests (from 1 up to 42 seats), making it unclear whether the observed price corresponds to the marginal seat or to an adjusted pricing structure.

The distinction between *fare1* and *fare2* is crucial for the econometric analysis, as it allows the comparison between censored and non-censored data and provides robustness in the estimation of pricing dynamics under different availability scenarios.

Additionally, zero values in both fare variables are treated as invalid observations rather than true prices. To avoid introducing bias into the analysis, these values are replaced with missing values (`NaN`).

In [9]:
df_final["fare1"] = df_final["fare1"].replace(0, np.nan)
df_final["fare2"] = df_final["fare2"].replace(0, np.nan)

## 3. Panel structure and identifier construction

In order to implement the econometric model, the dataset is structured as panel data. Each observation corresponds to a specific route and booking date, allowing the analysis of fare dynamics over time.

To define the panel dimension, a unique identifier is constructed by combining the route identifier (*routeID*) with the day of departure (*ddate*). This approach allows grouping observations that belong to the same route and departure date, ensuring consistency in the analysis of fare evolution.

This structure is consistent with the model specification, where fare dynamics are analyzed as a function of both capacity (available seats) and time to departure. By organizing the data in this way, it becomes possible to control for unobserved heterogeneity at the route level and track pricing behavior across different booking periods.

In [10]:
# Ensure date format
df_final["ddate"] = pd.to_datetime(df_final["ddate"])

# Create unique panel identifier (route + full date)
df_final["id"] = (
    df_final["routeID"].astype(str) + "_" + df_final["ddate"].dt.strftime("%Y%m%d")
)

## 4. Capacity-based feature engineering

A key dimension of the analysis is the role of capacity in fare determination. Following the empirical framework of the study, available seats are used as a central explanatory variable to understand how prices evolve as seat availability changes over time.

To complement this information, I construct the variable *capacity_reverse*, defined as the minimum number of available seats observed within each route–departure combination. This variable summarizes the lowest capacity level reached for a given trip and helps characterize the degree of seat scarcity within that booking window.

This transformation is consistent with the underlying pricing logic discussed in the paper, where fare adjustments are expected to respond to remaining capacity and demand conditions.

In [11]:
# Create a working copy of the dataset
df_final1 = df_final.copy()

# Minimum number of available seats within each route-departure group
df_final1["capacity_reverse"] = df_final1.groupby("id")["sleft"].transform("min")

## 5. Maximum fare within each route–departure combination

To further characterize pricing behavior, I construct the variable *topfare*, defined as the maximum fare observed within each route–departure combination.

This variable captures the upper bound of the fare distribution for a given trip and provides additional information on the extent of price variation within the same service. In the context of revenue management, this is useful for identifying how high fares can rise under specific capacity and booking conditions.

In [12]:
# Maximum observed fare within each route-departure group
df_final1["topfare"] = df_final1.groupby("id")["fare"].transform("max")

## 6. Fare distribution within each query

To describe the fare structure returned by each search, I compute summary statistics at the query level. Specifically, I construct the mean, median, and maximum fare observed within each *queryID*.

These measures help capture the distribution of prices associated with a given query and provide additional context for understanding how fares vary across booking conditions. They are useful descriptive indicators of the pricing environment observed on the platform.

In [13]:
# Fare summary statistics within each query
df_final1["mean_fare"] = df_final1.groupby("queryID")["fare"].transform("mean")
df_final1["median_fare"] = df_final1.groupby("queryID")["fare"].transform("median")
df_final1["max_fare"] = df_final1.groupby("queryID")["fare"].transform("max")

## 7. Group identifiers for routes and departure dates

To facilitate the analysis and modeling process, additional grouping variables are constructed.

First, a variable (*book*) is created to identify each departure date. This allows grouping observations that correspond to the same travel day and helps organize the temporal structure of the dataset.

Second, a route-level identifier (*route_group*) is defined to distinguish between different origin–destination pairs. This is particularly useful for capturing route-specific characteristics and enables a clearer comparison of pricing patterns across different routes.

These grouping variables support the panel data structure and improve the interpretability of the analysis.

In [14]:
# Create numeric group identifier by departure date
df_final1["book"] = df_final1.groupby("ddate").ngroup() + 1

# Create numeric group identifier by route
df_final1["route_group"] = df_final1.groupby("route").ngroup() + 1